# Gradient Boosting (Regression)

In [11]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error


# 1. Create a simple dataset

X = np.array([[1], [2], [3], [4], [5], [6]])
y = np.array([10, 20, 25, 40, 45, 60])

# 2. Initialize the model

# Start with the mean of all target values
initial_prediction = np.mean(y)

predictions = np.full(len(y), initial_prediction)

print("Actual values:       ", y)
print("Initial predictions: ", predictions)

# 3. Set hyperparameters

n_estimators = 1000
learning_rate = 0.1

trees = []

# 4. Train the boosting model

for i in range(n_estimators):

    # Calculate residuals
    residuals = y - predictions

    # Train a decision tree to predict residuals
    tree = DecisionTreeRegressor(
        max_depth=2,
        random_state=42
    )

    tree.fit(X, residuals)

    # Predict the correction
    correction = tree.predict(X)

    # Update predictions 
    predictions += learning_rate * correction

    # Save the trained tree
    trees.append(tree)

    # Calculate error
    mse = mean_squared_error(y, predictions)

    print(f"\nIteration {i + 1}")
    print("Residuals:  ", np.round(residuals, 2))
    print("Correction: ", np.round(correction, 2))
    print("Predictions:", np.round(predictions, 2))
    print("MSE:        ", round(mse, 2))

# 5. Make predictions on new data

X_new = np.array([[2.5], [4.5], [7]])

new_predictions = np.full(len(X_new), initial_prediction)

for tree in trees:
    new_predictions += learning_rate * tree.predict(X_new)

print("\nNew input values:", X_new.flatten())
print("Predicted values:", np.round(new_predictions, 2))

Actual values:        [10 20 25 40 45 60]
Initial predictions:  [33.33333333 33.33333333 33.33333333 33.33333333 33.33333333 33.33333333]

Iteration 1
Residuals:   [-23.33 -13.33  -8.33   6.67  11.67  26.67]
Correction:  [-23.33 -10.83 -10.83   9.17   9.17  26.67]
Predictions: [31.   32.25 32.25 34.25 34.25 36.  ]
MSE:         228.04

Iteration 2
Residuals:   [-21.   -12.25  -7.25   5.75  10.75  24.  ]
Correction:  [-21.    -9.75  -9.75   8.25   8.25  24.  ]
Predictions: [28.9  31.28 31.28 35.08 35.08 38.4 ]
MSE:         185.51

Iteration 3
Residuals:   [-18.9  -11.27  -6.27   4.92   9.92  21.6 ]
Correction:  [-18.9   -8.77  -8.77   7.42   7.42  21.6 ]
Predictions: [27.01 30.4  30.4  35.82 35.82 40.56]
MSE:         151.05

Iteration 4
Residuals:   [-17.01 -10.4   -5.4    4.18   9.18  19.44]
Correction:  [-17.01  -7.9   -7.9    6.68   6.68  19.44]
Predictions: [25.31 29.61 29.61 36.49 36.49 42.5 ]
MSE:         123.14

Iteration 5
Residuals:   [-15.31  -9.61  -4.61   3.51   8.51  17.5 ]


In [12]:
print(X)

print(y)

[[1]
 [2]
 [3]
 [4]
 [5]
 [6]]
[10 20 25 40 45 60]


# GBM (Classification)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import log_loss

class GradientBoostingClassifier:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []

    def sigmoid(self, scores):
        scores = np.clip(scores, -500, 500)
        return 1 / (1 + np.exp(-scores))

    def fit(self, X, y):
        # Initial prediction: log odds of the positive class
        p = np.clip(np.mean(y), 1e-10, 1 - 1e-10)
        initial_score = np.log(p / (1 - p))
        self.initial_score = initial_score

        print("p:", p)
        print("1-p:", 1-p)
        print("p/1-p:", p/1-p)
        print("initial_score:", initial_score )

        scores = np.full(len(y), initial_score)
        self.trees = []
    
        for i in range(self.n_estimators):
            # Convert raw scores into probabilities
            probabilities = self.sigmoid(scores)

            # Negative gradient of binary logistic loss
            residuals = y - probabilities

            # Fit a regression tree to the correction targets
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=42
            )
            tree.fit(X, residuals)

            # Update raw scores
            correction = tree.predict(X)
            scores += self.learning_rate * correction

            self.trees.append(tree)

            # Evaluate classification loss
            loss = log_loss(y, self.sigmoid(scores))
            print(f"Iteration {i + 1}: Log Loss = {loss:.4f}")

    def predict_proba(self, X):
        scores = np.full(X.shape[0], self.initial_score)

        for tree in self.trees:
            scores += self.learning_rate * tree.predict(X)

        probabilities = self.sigmoid(scores)

        # Return probabilities for [class 0, class 1]
        return np.column_stack([1 - probabilities, probabilities])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# Sample dataset

# Features: [Hours studied, Hours slept]
X = np.array([
    [1, 5],
    [2, 6],
    [2, 4],
    [3, 7],
    [4, 6],
    [5, 8],
    [6, 7],
    [7, 8],
    [8, 6],
    [9, 8]
], dtype=float)

# 0 = Fail, 1 = Pass
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])


# Train

model = GradientBoostingClassifier(
    n_estimators=10,
    learning_rate=0.1,
    max_depth=2
)

model.fit(X, y)


# Test

X_test = np.array([
    [2, 5],
    [5, 7],
    [8, 7]
], dtype=float)

probabilities = model.predict_proba(X_test)
predictions = model.predict(X_test)

print("\nFinal predictions:")

for features, probability, prediction in zip(
    X_test, probabilities[:, 1], predictions
):
    result = "Pass" if prediction == 1 else "Fail"
    print(
        f"Features: {features}, "
        f"Pass probability: {probability:.3f}, "
        f"Prediction: {result}"
    )

p: 0.5
1-p: 0.5
p/1-p: 0.0
0.0
Iteration 1: Log Loss = 0.6685
Iteration 2: Log Loss = 0.6450
Iteration 3: Log Loss = 0.6227
Iteration 4: Log Loss = 0.6015
Iteration 5: Log Loss = 0.5813
Iteration 6: Log Loss = 0.5621
Iteration 7: Log Loss = 0.5438
Iteration 8: Log Loss = 0.5264
Iteration 9: Log Loss = 0.5099
Iteration 10: Log Loss = 0.4941

Final predictions:
Features: [2. 5.], Pass probability: 0.390, Prediction: Fail
Features: [5. 7.], Pass probability: 0.610, Prediction: Pass
Features: [8. 7.], Pass probability: 0.610, Prediction: Pass
